In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('vuelos_10k.csv')


# Fase 1:

In [ ]:
#Valores nulos
df = df.dropna(subset=['Origen', 'Destino'])

#Formato inconsistente
df['Destino'] = df['Destino'].apply(lambda x: x.capitalize())
df['Origen'] = df['Origen'].apply(lambda x: x.capitalize())

#Anomalias logicas
df = df[df['Origen'] != df['Destino']]

#Errores de captura (distancia negativa)
df['Distancia_km'] = df['Distancia_km'].abs()

#Duplicados (rutas unicas)
df = df.drop_duplicates(subset=['Origen', 'Destino'])


# Fase 2:

Importen la librería networkx.

Utilicen la función nx.from_pandas_edgelist() para convertir su DataFrame en un Grafo Dirigido (nx.DiGraph).

Impriman en pantalla cuántos Nodos (Aeropuertos) y cuántas Aristas (Rutas) tiene su grafo final.

In [ ]:
!pip install networkx

In [ ]:
G = nx.from_pandas_edgelist(df, source='Origen', target='Destino', edge_attr=None, create_using=nx.DiGraph)

In [ ]:
print(f"Nodos (Aeropuertos): {G.number_of_nodes()}")
print(f"Aristas (Rutas): {G.number_of_edges()}")

Nodos (Aeropuertos): 3253
Aristas (Rutas): 9743


# Fase 3:

¡Llegó la hora de comprobar la teoría de la clase!

Construyan la Matriz de Adyacencia densa a partir de su grafo usando nx.adjacency_matrix(G).todense().
Utilicen código Python (ej. el atributo .nbytes) para calcular cuántos Megabytes (MB) de Memoria RAM está consumiendo su Matriz de Adyacencia llena de ceros.
Calculen cuántos Megabytes está consumiendo su DataFrame limpio (la Lista de Aristas).
Veredicto: Añadan una celda Markdown al final de su notebook respondiendo: ¿Cuántas veces más pesada es la Matriz en comparación con la Lista de Aristas y por qué sucede esto?

In [ ]:
A = nx.adjacency_matrix(G).todense()
A = np.array(A)

In [ ]:
matriz_bytes = A.nbytes
matriz_mb = matriz_bytes / (1024 * 1024)

df_bytes = df.memory_usage(deep=True).sum()
df_mb = df_bytes / (1024 * 1024)

print(f"Consumo de Memoria RAM de la Matriz de Adyacencia densa: {matriz_mb:.4f} MB")
print(f"Consumo de Memoria RAM del Dataframe limpio (lista de aristas): {df_mb:.4f} MB")

Consumo de Memoria RAM de la Matriz de Adyacencia densa: 80.7343 MB
Consumo de Memoria RAM del Dataframe limpio (lista de aristas): 1.1150 MB


# Veredicto

### Veredicto: ¿Cuántas veces más pesada es la Matriz en comparación con la Lista de Aristas y por qué sucede esto?

Basándonos en los resultados obtenidos:
- **Consumo de la Matriz de Adyacencia densa:** ~80.73 MB
- **Consumo del DataFrame limpio (Lista de Aristas):** ~1.12 MB

#### 1. ¿Cuántas veces más pesada es la Matriz?
La Matriz de Adyacencia densa es aproximadamente **72 veces más pesada** que la Lista de Aristas (DataFrame).

$$\frac{80.7343 \text{ MB}}{1.1150 \text{ MB}} \approx 72.41$$

#### 2. ¿Por qué sucede esto?
* **Matriz de Adyacencia Densa ($O(V^2)$):** Al convertir el grafo en una matriz densa, se reserva un espacio en memoria para representar las conexiones entre todos los pares posibles de nodos. Dado que tenemos 3,253 nodos, la matriz resultante tiene un tamaño de $3253 \times 3253 = 10,582,009$ elementos. Cada celda de esta matriz almacena un valor (generalmente un entero de 64 bits u 8 bytes), sin importar si existe o no una ruta entre los aeropuertos. Como la inmensa mayoría de los aeropuertos no están conectados directamente (grafo disperso), la matriz queda llena de ceros que consumen memoria innecesariamente.
* **Lista de Aristas ($O(E)$):** El DataFrame de pandas solo almacena las relaciones existentes (las aristas reales, que son 9,743 rutas). No gasta memoria en representar la ausencia de vuelos entre aeropuertos. Por ello, es una estructura muchísimo más eficiente para grafos dispersos.